# CBIS-DDSM → DeepProbLog (single Run All)

This notebook is the full pipeline: **prepare the CSVs**, **load mammogram JPEGs**, **train DeepProbLog**, then **evaluate and show images**.

**What it does**
1. Merge mass and calcification tables and build a patient-disjoint train / val / test split.
2. Map CSV descriptors to concepts (`kind`, `shape`, `margin`, `calc`).
3. Load the cropped JPEG for each row (or stop if a file is missing).
4. Train neural concept nets plus a Prolog program that queries `malignant(X)`.
5. Report accuracy, F1, and concept checks.
6. Draw train / validation / **test** image grids, then draw test images again with predictions.

**Kaggle setup**
- Attach dataset: `cbis-ddsm-breast-cancer-image-dataset`
- Accelerator: GPU T4 (P100 is not compatible with this PyTorch build)
- First check: `SMOKE_TEST = True` (tiny sample, fast wiring test)
- Full run: `SMOKE_TEST = False`

**How to read the log**
- `DEVICE: cuda` means the GPU is in use.
- `loaded N tensors (0 synthetic)` means real JPEGs were found. If synthetic is not 0, the images are placeholders and the run is only a smoke test.

The Prolog rules are a computational working program, not official BI-RADS criteria.


## 1) Install packages

**What:** install DeepProbLog, scikit-learn, pandas, and Pillow in this kernel.

**Why:** Kaggle runtimes do not always include DeepProbLog. This cell is safe to re-run.


In [ ]:
%pip install -q deepproblog scikit-learn pandas pillow

## 2) Imports

**What:** load libraries for tables, images, PyTorch, and plots.

**Why:** later cells need pandas for CSVs, matplotlib to show mammograms, and torch for the concept networks.


In [ ]:
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torchvision import transforms

print("pandas:", pd.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3) Config

**What:** one settings cell for the whole notebook.

**Why:** change `SMOKE_TEST`, epochs, batch size, image size, and `SHOW_N_IMAGES` here.
- `SMOKE_TEST=True` — tiny wiring test (few patients, 1 epoch).
- `SMOKE_TEST=False` — full split and 15 epochs.
- `USE_SYNTHETIC_IF_MISSING=False` — missing JPEGs must raise an error.
- `FORCE_CPU=False` — use T4 if the CUDA probe succeeds.


In [ ]:
# ===================== ONE CONFIG FOR EVERYTHING =====================
SMOKE_TEST = False                 # True = tiny wiring test; False = full run
SMOKE_N_PATIENTS = 20             # patients per official split (data prep)
SMOKE_N_SAMPLES = 24              # max rows used for DeepProbLog tensors
VAL_SIZE = 0.15
RANDOM_SEED = 42

EPOCHS = 1 if SMOKE_TEST else 15         # full run: 15 epochs
BATCH_SIZE = 2 if SMOKE_TEST else 4      # 4 on T4; drop to 2 if CUDA runs out of memory
IMAGE_SIZE = 64 if SMOKE_TEST else 128   # 128 sees more of the lesion than 64
LR = 1e-3
USE_SYNTHETIC_IF_MISSING = False  # missing JPEG must fail; do not invent images
FORCE_CPU = False                 # use GPU if the probe succeeds (T4). P100 still falls back to CPU.
SHOW_N_IMAGES = 8                 # pictures to draw in each image grid

DATASET_HANDLE = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"
KAGGLE_INPUT = Path("/kaggle/input/cbis-ddsm-breast-cancer-image-dataset")
KAGGLE_WORKING = Path("/kaggle/working")
LOCAL_DATA = Path("../data")

CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}

PROGRAM_FALLBACK = """
nn(kind_net, [X], Y, [mass, calcification]) :: lesion_kind(X, Y).
nn(shape_net, [X], Y, [regular, irregular, unknown]) :: shape(X, Y).
nn(margin_net, [X], Y, [circumscribed, spiculated, other, unknown]) :: margin(X, Y).
nn(calc_net, [X], Y, [absent, typically_benign, suspicious]) :: calc(X, Y).

malignant(X) :- lesion_kind(X, mass), shape(X, irregular).
malignant(X) :- lesion_kind(X, mass), margin(X, spiculated).
malignant(X) :- lesion_kind(X, calcification), calc(X, suspicious).
""".strip()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def pick_device(force_cpu=False):
    """Prefer CUDA, but fall back to CPU if the binary can't run on this GPU.
    Common on Kaggle when PyTorch wheels don't match the accelerator arch.
    """
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        # Probe that kernels actually work on this GPU (not just "cuda available").
        x = torch.zeros(1, device="cuda")
        y = x + 1
        torch.cuda.synchronize()
        del x, y
        torch.cuda.empty_cache()
        return torch.device("cuda")
    except Exception as exc:
        print("CUDA probe failed → using CPU. Reason:", type(exc).__name__, str(exc)[:200])
        return torch.device("cpu")

DEVICE = pick_device(FORCE_CPU)
print("DEVICE:", DEVICE)
if DEVICE.type == "cpu" and torch.cuda.is_available():
    print("Note: CUDA is visible but unusable with this PyTorch build. DeepProbLog will run on CPU.")

if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working/processed")
else:
    OUTPUT_DIR = Path("../notebooks/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    print("=" * 64)
    print("SMOKE_TEST=True — tiny wiring test on a small sample.")
    print("=" * 64)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 4) Find the four CBIS CSVs

**What:** search Kaggle input, then local `data/`, for the official mass and calc description files.

**Why:** the files are not always at a fixed path. Walking the tree (skipping `jpeg/`) keeps this cell working on Kaggle and on a laptop.


In [ ]:
def find_csv_paths(root, filenames):
    root = Path(root)
    found, remaining = {}, set(filenames.values())
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
        for fname in files:
            if fname in remaining:
                key = next(k for k, v in filenames.items() if v == fname)
                found[key] = Path(dirpath) / fname
                remaining.discard(fname)
        if not remaining:
            break
    missing = [k for k in filenames if k not in found]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {root}")
    return found


def resolve_data_root():
    if KAGGLE_INPUT.is_dir():
        return KAGGLE_INPUT, "kaggle_input"
    # other common Kaggle mount
    for p in Path("/kaggle/input").glob("*cbis*") if Path("/kaggle/input").is_dir() else []:
        if p.is_dir():
            return p, "kaggle_input_scan"
    if LOCAL_DATA.is_dir() and all((LOCAL_DATA / fn).exists() for fn in CSV_FILENAMES.values()):
        return LOCAL_DATA, "local_data"
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        return downloaded, "kagglehub_download"
    except Exception as exc:
        raise FileNotFoundError("No CBIS-DDSM data found") from exc


DATA_ROOT, DATA_SOURCE = resolve_data_root()
CSV_PATHS = find_csv_paths(DATA_ROOT, CSV_FILENAMES)
print(f"Data source: {DATA_SOURCE}\nRoot: {DATA_ROOT}")
for k, v in CSV_PATHS.items():
    print(f"  {k}: {v}")

## 5) Merge mass+calc, fix leakage, split by patient

**What:** concatenate mass and calcification tables, assign overlapping patients entirely to train, then cut a patient-disjoint validation set.

**Why:** after merging, some patients appear in both official train and test. A patient-level split is required so the same woman is never in train and test.


In [ ]:
STRING_COLUMNS = [
    "patient_id", "left or right breast", "image view", "abnormality type",
    "mass shape", "mass margins", "calc type", "calc distribution", "pathology",
    "image file path", "cropped image file path", "ROI mask file path",
]


def normalize_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    for col in STRING_COLUMNS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out


def add_meta(df, kind, official_split):
    out = df.copy()
    out["abnormality_kind"] = kind
    out["official_split"] = official_split
    return out


def sample_patients(df, n_patients, seed):
    patients = df["patient_id"].dropna().unique()
    if len(patients) <= n_patients:
        return df.copy()
    chosen = set(random.Random(seed).sample(list(patients), n_patients))
    return df[df["patient_id"].isin(chosen)].copy()


def resolve_cross_split_patient_leakage(train_df, test_df):
    """CBIS mass/calc official CSVs can place the same patient in train AND test
    (e.g. mass in train, calc in test). Keep ALL rows for those patients in train
    and drop them from test so splits stay patient-disjoint.
    """
    train_p = set(train_df["patient_id"].dropna())
    test_p = set(test_df["patient_id"].dropna())
    overlap = train_p & test_p
    if not overlap:
        print("No cross-split patient overlap.")
        return train_df, test_df

    print(f"Found {len(overlap)} patients in BOTH official train and test after merge.")
    print("  Assigning all their rows to TRAIN and removing them from TEST.")
    leaked_test_rows = test_df[test_df["patient_id"].isin(overlap)].copy()
    train_fixed = pd.concat([train_df, leaked_test_rows], ignore_index=True)
    test_fixed = test_df[~test_df["patient_id"].isin(overlap)].reset_index(drop=True)
    # drop duplicate rows if any exact duplicates appear
    train_fixed = train_fixed.drop_duplicates().reset_index(drop=True)
    return train_fixed, test_fixed


raw = {k: pd.read_csv(v) for k, v in CSV_PATHS.items()}
mass_train = add_meta(normalize_columns(raw["mass_train"]), "mass", "train")
mass_test = add_meta(normalize_columns(raw["mass_test"]), "mass", "test")
calc_train = add_meta(normalize_columns(raw["calc_train"]), "calcification", "train")
calc_test = add_meta(normalize_columns(raw["calc_test"]), "calcification", "test")

df_train_all = pd.concat([mass_train, calc_train], ignore_index=True)
df_test_all = pd.concat([mass_test, calc_test], ignore_index=True)
print("Official train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("Official test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

df_train_all, df_test_all = resolve_cross_split_patient_leakage(df_train_all, df_test_all)
print("After leakage fix train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("After leakage fix test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

if SMOKE_TEST:
    df_train_all = sample_patients(df_train_all, SMOKE_N_PATIENTS, RANDOM_SEED)
    df_test_all = sample_patients(df_test_all, SMOKE_N_PATIENTS, RANDOM_SEED + 1)
    print("After smoke sampling:", len(df_train_all), "train rows |", len(df_test_all), "test rows")

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
tr_idx, va_idx = next(gss.split(df_train_all, groups=df_train_all["patient_id"]))
df_train = df_train_all.iloc[tr_idx].reset_index(drop=True)
df_val = df_train_all.iloc[va_idx].reset_index(drop=True)
df_test = df_test_all.reset_index(drop=True)

tr_p, va_p, te_p = set(df_train.patient_id), set(df_val.patient_id), set(df_test.patient_id)
print("overlap train∩val:", len(tr_p & va_p), "| train∩test:", len(tr_p & te_p), "| val∩test:", len(va_p & te_p))
assert tr_p.isdisjoint(va_p), "Train/val patient leakage"
assert tr_p.isdisjoint(te_p), "Train/test patient leakage"
assert va_p.isdisjoint(te_p), "Val/test patient leakage"
print("Leakage checks passed.")
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{name:5s}: {len(df):>5} rows | {df['patient_id'].nunique():>4} patients")


## 6) Save the split tables

**What:** write `cbis_train.csv`, `cbis_val.csv`, `cbis_test.csv`, and a patient-to-split map.

**Why:** so you can inspect exactly which cases this run used.


In [ ]:
# Save processed artifacts (same session will also train on these frames)
patient_splits = []
for split_name, frame in (("train", df_train), ("val", df_val), ("test", df_test)):
    for pid in sorted(frame["patient_id"].dropna().unique()):
        patient_splits.append({"patient_id": pid, "split": split_name})
pd.DataFrame(patient_splits).to_csv(OUTPUT_DIR / "cbis_patient_splits.csv", index=False)
df_train.to_csv(OUTPUT_DIR / "cbis_train.csv", index=False)
df_val.to_csv(OUTPUT_DIR / "cbis_val.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "cbis_test.csv", index=False)

meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "smoke_n_patients": SMOKE_N_PATIENTS if SMOKE_TEST else None,
    "val_size": VAL_SIZE,
    "random_seed": RANDOM_SEED,
    "data_source": DATA_SOURCE,
    "data_root": str(DATA_ROOT),
    "counts": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_patients": df_train["patient_id"].nunique(),
        "val_patients": df_val["patient_id"].nunique(),
        "test_patients": df_test["patient_id"].nunique(),
    },
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
print("Saved prep outputs to", OUTPUT_DIR.resolve())

## 7) Map CSV descriptors to concepts

**What:** convert `mass shape`, `mass margins`, `calc type`, and `abnormality_kind` into the symbols used in the Prolog program.

**Why:** DeepProbLog queries `shape(X, irregular)`, not the original CBIS strings. Any calcification is **not** treated as malignant; only suspicious types map to `suspicious`.


In [ ]:
def tokenize_descriptor(value):
    if pd.isna(value):
        return []
    text = str(value).strip().upper()
    if text in {"", "NAN", "NONE"}:
        return []
    for sep in ("-", "/", "_", " "):
        text = text.replace(sep, " ")
    return [t for t in text.split() if t]


def map_shape(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if any(t in {"IRREGULAR", "ARCHITECTURAL", "DISTORTION"} for t in tokens):
        return "irregular"
    if any(t in {"ROUND", "OVAL", "LOBULATED"} for t in tokens):
        return "regular"
    return "unknown"


def map_margin(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if "SPICULATED" in tokens:
        return "spiculated"
    if "CIRCUMSCRIBED" in tokens:
        return "circumscribed"
    return "other"


def map_calc(row):
    # Any calcification present is NOT a malignancy label. Most CBIS calc cases are benign.
    if str(row.get("abnormality_kind", "")).lower() != "calcification":
        return "absent"
    tokens = tokenize_descriptor(row.get("calc type"))
    if any(t in {"PLEOMORPHIC", "AMORPHOUS"} for t in tokens) or (
        "FINE" in tokens and "LINEAR" in tokens
    ) or "BRANCHING" in tokens:
        return "suspicious"
    return "typically_benign"


def add_concepts(df):
    out = df.copy()
    out["shape_concept"] = out["mass shape"].map(map_shape) if "mass shape" in out.columns else "unknown"
    out["margin_concept"] = out["mass margins"].map(map_margin) if "mass margins" in out.columns else "unknown"
    out["calc_concept"] = out.apply(map_calc, axis=1)
    out["kind_concept"] = out["abnormality_kind"].map(
        lambda x: "calcification" if str(x).strip().lower() == "calcification" else "mass"
    )
    out["malignant"] = out["pathology"].map(lambda x: 1.0 if str(x).strip().upper() == "MALIGNANT" else 0.0)
    return out


df_train = add_concepts(df_train)
df_val = add_concepts(df_val)
df_test = add_concepts(df_test)
display(df_train[["patient_id", "abnormality_kind", "shape_concept", "margin_concept", "calc_concept", "malignant"]].head())
print("calc_concept counts:\n", df_train["calc_concept"].value_counts())
print("malignant rate train/val/test:",
      float(df_train["malignant"].mean()),
      float(df_val["malignant"].mean()),
      float(df_test["malignant"].mean()))

## 8) Load JPEGs and show train / val / test images

**What:** resolve each CSV path to a JPEG folder (by DICOM UID), prefer a gray-level photo over a binary mask, materialize tensors, then **draw image grids** for train, validation, and test.

**Why:** you need to see the actual mammogram crops. If every picture is a placeholder, JPEG matching failed. `synthetic` must be 0 for a real-image run.


In [ ]:
def find_jpeg_root():
    candidates = [DATA_ROOT / "jpeg", KAGGLE_INPUT / "jpeg", LOCAL_DATA / "jpeg"]
    if Path("/kaggle/input").is_dir():
        candidates.extend(p for p in Path("/kaggle/input").rglob("jpeg") if p.is_dir())
    for p in candidates:
        if p.is_dir():
            return p
    return None


JPEG_ROOT = find_jpeg_root()
print("JPEG root:", JPEG_ROOT)
print("Tensors will be placed on:", DEVICE)

tfm = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


def clean_relpath(value):
    if pd.isna(value):
        return None
    text = str(value).strip().strip('"').split("\n")[0].strip()
    return text or None


_JPEG_INDEX = None


def jpeg_folder_index():
    """Map folder names that actually contain images to their paths.

    The awsaf49 jpeg tree is not the CSV DICOM path. Images usually live in a
    folder named by the series UID (3rd path component), as 1-1.jpg / 1-2.jpg.
    """
    global _JPEG_INDEX
    if _JPEG_INDEX is not None:
        return _JPEG_INDEX
    index = {}
    if JPEG_ROOT is None or not JPEG_ROOT.is_dir():
        _JPEG_INDEX = index
        return index
    for dirpath, _dirnames, filenames in os.walk(JPEG_ROOT):
        imgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if not imgs:
            continue
        folder = Path(dirpath)
        index[folder.name] = folder
    _JPEG_INDEX = index
    print(f"Indexed {len(index)} jpeg folders")
    print("example jpeg folders:", list(index)[:5])
    return index


def image_photo_score(path):
    """Higher score = more like a mammogram crop, less like a binary ROI mask."""
    with Image.open(path) as im:
        arr = np.asarray(im.convert("L"), dtype=np.uint8)
    n_unique = int(len(np.unique(arr)))
    mid_gray = float(((arr > 15) & (arr < 240)).mean())
    return (n_unique, mid_gray, float(arr.std()))


def pick_image(folder, prefer_crop=True):
    folder = Path(folder)
    jpgs = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.jpeg")) + sorted(folder.glob("*.png"))
    if not jpgs:
        jpgs = sorted(folder.rglob("*.jpg")) + sorted(folder.rglob("*.jpeg")) + sorted(folder.rglob("*.png"))
    if not jpgs:
        return None
    if len(jpgs) == 1:
        return jpgs[0]
    # CBIS crop/mask folders contain 1-1.jpg and 1-2.jpg, and the CSV columns
    # are often swapped. Prefer the file with many gray levels (photo), not a mask.
    return max(jpgs, key=image_photo_score)


def resolve_jpeg(row):
    if JPEG_ROOT is None:
        return None
    index = jpeg_folder_index()
    for col in ("cropped image file path", "image file path"):
        rel = clean_relpath(row.get(col))
        if not rel:
            continue
        parts = [p for p in rel.replace("\\", "/").split("/") if p]
        if not parts:
            continue
        prefer_crop = col.startswith("cropped")
        exact = JPEG_ROOT / rel.replace(".dcm", ".jpg")
        if exact.is_file():
            return exact
        keys = []
        # Do not use Path.stem here. DICOM UIDs contain dots, so stem()
        # would chop the series id and never match the jpeg folder name.
        for part in parts:
            key = part[:-4] if part.lower().endswith(".dcm") else part
            if key:
                keys.append(key)
        seen = set()
        for key in keys:
            if not key or key in seen:
                continue
            seen.add(key)
            folder = index.get(key)
            if folder is None and (JPEG_ROOT / key).is_dir():
                folder = JPEG_ROOT / key
            if folder is None:
                continue
            img = pick_image(folder, prefer_crop=prefer_crop)
            if img is not None:
                return img
    return None


def _to_device(t):
    """Move tensor to DEVICE; if CUDA fails mid-run, switch globally to CPU."""
    global DEVICE
    try:
        return t.to(DEVICE)
    except Exception as exc:
        print("Device move failed (", type(exc).__name__, ") → forcing CPU for the rest of the run.")
        DEVICE = torch.device("cpu")
        return t.to(DEVICE)


def concept_placeholder(row):
    """Used only when a JPEG is missing. These placeholders are for a smoke test, not a real-image run."""
    shape_c = row.get("shape_concept", "unknown")
    margin_c = row.get("margin_concept", "unknown")
    calc_c = row.get("calc_concept", "absent")
    t = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
    third = IMAGE_SIZE // 3
    t[0, :third, :] = {"irregular": 1.0, "regular": 0.55, "unknown": 0.15}.get(shape_c, 0.15)
    t[0, third:2 * third, :] = {"spiculated": 1.0, "circumscribed": 0.55, "other": 0.35, "unknown": 0.15}.get(margin_c, 0.15)
    t[0, 2 * third:, :] = {"suspicious": 1.0, "typically_benign": 0.55, "absent": 0.15}.get(calc_c, 0.15)
    return t


def load_tensor(row):
    path = resolve_jpeg(row)
    if path is not None and path.is_file():
        with Image.open(path) as img:
            tensor = _to_device(tfm(img.convert("L")))
            return tensor, str(path), False
    if not USE_SYNTHETIC_IF_MISSING:
        rel = clean_relpath(row.get("cropped image file path")) or clean_relpath(row.get("image file path"))
        raise FileNotFoundError(
            f"No JPEG for patient {row.get('patient_id')}. csv_path={rel!r}. "
            f"JPEG_ROOT={JPEG_ROOT}. Check the 'Indexed N jpeg folders' line above."
        )
    return _to_device(concept_placeholder(row)), None, True


def materialize_split(df, n_limit=None):
    if n_limit is not None:
        df = df.head(n_limit).copy()
    tensors, records, n_synth = [], [], 0
    for _, row in df.iterrows():
        tensor, path, synthetic = load_tensor(row)
        tensors.append(tensor)
        rec = row.to_dict()
        rec["resolved_path"] = path
        rec["synthetic"] = synthetic
        records.append(rec)
        n_synth += int(synthetic)
    print(f"  loaded {len(tensors)} tensors ({n_synth} synthetic) on {DEVICE}")
    return tensors, pd.DataFrame(records)


limit = SMOKE_N_SAMPLES if SMOKE_TEST else None
print("Materializing train"); train_tensors, df_train = materialize_split(df_train, limit)
print("Materializing val");   val_tensors, df_val = materialize_split(df_val, limit)
print("Materializing test");  test_tensors, df_test = materialize_split(df_test, limit)
if df_train["synthetic"].all():
    print("WARNING: all train images are synthetic placeholders — check JPEG_ROOT on Kaggle.")
    print("This is a smoke test only until real JPEGs load (synthetic count must be 0).")
elif df_train["synthetic"].any():
    print("WARNING: some train images are synthetic placeholders.")

def tensor_to_numpy(t):
    x = t.detach().cpu() if hasattr(t, "detach") else t
    if hasattr(x, "numpy"):
        x = x.numpy()
    x = np.squeeze(x)
    return x


def show_image_grid(title, df, tensors, n=None):
    n = min(int(n if n is not None else SHOW_N_IMAGES), len(df), len(tensors))
    if n == 0:
        print(title, ": no images")
        return
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        ax = axes[i]
        ax.imshow(tensor_to_numpy(tensors[i]), cmap="gray")
        row = df.iloc[i]
        synth = " [placeholder]" if bool(row.get("synthetic", False)) else ""
        ax.set_title(
            f"{row['patient_id']}{synth}\n{row['pathology']}\n{row['abnormality_kind']}",
            fontsize=8,
        )
        ax.axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


print("\nShowing loaded images (real JPEG if synthetic=0):")
show_image_grid("Train images", df_train, train_tensors)
show_image_grid("Validation images", df_val, val_tensors)
show_image_grid("Test images", df_test, test_tensors)


## 9) Neural nets and DeepProbLog datasets

**What:** define four tiny CNNs (kind, shape, margin, calc) and wrap each row as a `malignant` query plus concept queries.

**Why:** concept queries teach the nets the CSV labels; the `malignant` query teaches the combination that the rules use.


In [ ]:
class ConceptCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Linear(64 * 4 * 4, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # DeepProbLog may stack CPU tensors even if module is on CUDA — force match.
        if not torch.is_tensor(x):
            x = torch.as_tensor(x)
        x = x.to(next(self.parameters()).device, dtype=torch.float32)
        if x.dim() == 3:  # [C,H,W] -> [1,C,H,W]
            x = x.unsqueeze(0)
        return self.softmax(self.classifier(self.net(x).flatten(1)))


shape_net = ConceptCNN(3).to(DEVICE)
margin_net = ConceptCNN(4).to(DEVICE)
calc_net = ConceptCNN(3).to(DEVICE)
kind_net = ConceptCNN(2).to(DEVICE)
print("Networks on:", next(shape_net.parameters()).device)

from problog.logic import Term, Constant
from deepproblog.dataset import Dataset, DataLoader, QueryDataset
from deepproblog.query import Query


class TensorSource:
    def __init__(self, tensors, device=None):
        self.tensors = tensors
        self.device = device

    def __getitem__(self, item):
        t = self.tensors[int(item[0])]
        if self.device is not None:
            t = t.to(self.device)
        return t


class BreastQueryDataset(Dataset):
    def __init__(self, split_name, df):
        self.split_name = split_name
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _tensor_term(self, i):
        return Term("tensor", Term(self.split_name, Constant(i)))

    def to_query(self, i):
        row = self.df.iloc[i]
        return Query(Term("malignant", self._tensor_term(i)), p=float(row["malignant"]))

    def concept_queries(self):
        qs = []
        for i, row in self.df.iterrows():
            img = self._tensor_term(i)
            qs.append(Query(Term("shape", img, Term(row["shape_concept"])), p=1.0))
            qs.append(Query(Term("margin", img, Term(row["margin_concept"])), p=1.0))
            qs.append(Query(Term("calc", img, Term(row["calc_concept"])), p=1.0))
            qs.append(Query(Term("lesion_kind", img, Term(row["kind_concept"])), p=1.0))
        return qs


train_set = BreastQueryDataset("train", df_train)
val_set = BreastQueryDataset("val", df_val)
test_set = BreastQueryDataset("test", df_test)
train_queries = QueryDataset(train_set.to_queries() + train_set.concept_queries())
print("example:", train_set.to_query(0))
print("n train queries (malignant + concepts):", len(train_queries))


## 10) Write the Prolog program and train

**What:** write `breast_diagnosis.pl` from the config string (so an old Kaggle file is not reused) and train DeepProbLog.

**Why:** mass evidence is used only when `lesion_kind` is mass. Suspicious calc is used only when `lesion_kind` is calcification.


In [ ]:
from deepproblog.engines import ExactEngine
from deepproblog.model import Model
from deepproblog.network import Network
from deepproblog.train import train_model

# Always write the notebook program. Do not reuse a stale Kaggle .pl from a previous run.
program_path = OUTPUT_DIR / "breast_diagnosis.pl"
program_path.write_text(PROGRAM_FALLBACK + "\n")
print("Using program:", program_path.resolve())
print(program_path.read_text())

shape_w = Network(shape_net, "shape_net", batching=True)
margin_w = Network(margin_net, "margin_net", batching=True)
calc_w = Network(calc_net, "calc_net", batching=True)
kind_w = Network(kind_net, "kind_net", batching=True)
shape_w.optimizer = torch.optim.Adam(shape_net.parameters(), lr=LR)
margin_w.optimizer = torch.optim.Adam(margin_net.parameters(), lr=LR)
calc_w.optimizer = torch.optim.Adam(calc_net.parameters(), lr=LR)
kind_w.optimizer = torch.optim.Adam(kind_net.parameters(), lr=LR)

model = Model(str(program_path), [shape_w, margin_w, calc_w, kind_w])
model.set_engine(ExactEngine(model), cache=False)
model.add_tensor_source("train", TensorSource(train_tensors, device=DEVICE))
model.add_tensor_source("val", TensorSource(val_tensors, device=DEVICE))
model.add_tensor_source("test", TensorSource(test_tensors, device=DEVICE))

loader = DataLoader(train_queries, BATCH_SIZE, shuffle=True, seed=RANDOM_SEED)
print(f"Training DeepProbLog: epochs={EPOCHS}, batch={BATCH_SIZE}, queries={len(train_queries)}")
train_model(model, loader, EPOCHS, log_iter=max(1, len(loader)))
print("Training finished.")

## 11) Metrics, reasoning traces, and predicted test images

**What:** score validation and test (accuracy, precision, F1, specificity), print concept probabilities, then draw **validation and test images with predictions**. Green title = match; red title = mismatch.

**Why:** F1 catches a model that always answers one class. The image grid is the visual check that `P(malignant)` is not the same on every picture.


In [ ]:
def extract_prob(result):
    if result is None:
        return 0.0
    mapping = result.result if hasattr(result, "result") else (result if isinstance(result, dict) else {})
    if not mapping:
        return 0.0
    vals = []
    for v in mapping.values():
        if hasattr(v, "detach"):
            vals.append(float(v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().mean()))
        else:
            vals.append(float(v))
    return float(sum(vals) / len(vals))


def malignant_probability(split_name, i):
    q = Query(Term("malignant", Term("tensor", Term(split_name, Constant(i)))))
    solved = model.solve([q])
    return extract_prob(solved[0]) if solved else 0.0


def _f1(tp, fp, fn):
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    if prec + rec == 0:
        return 0.0
    return 2.0 * prec * rec / (prec + rec)


def evaluate_split(name, dataset, df):
    y_true, y_prob = [], []
    for i in range(len(dataset)):
        y_true.append(int(df.iloc[i]["malignant"]))
        y_prob.append(malignant_probability(name, i))
    y_true, y_prob = np.array(y_true), np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    f1_mal = _f1(tp, fp, fn)
    f1_ben = _f1(tn, fn, fp)
    return {
        "split": name, "n": len(y_true),
        "accuracy": (tp + tn) / max(len(y_true), 1),
        "precision": precision,
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "f1_malignant": f1_mal,
        "f1_benign": f1_ben,
        "f1_macro": 0.5 * (f1_mal + f1_ben),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "mean_p": float(y_prob.mean()),
        "pct_pred_malignant": float(y_pred.mean()),
    }, y_prob


metrics_val, val_prob = evaluate_split("val", val_set, df_val)
metrics_test, test_prob = evaluate_split("test", test_set, df_test)
metrics_df = pd.DataFrame([metrics_val, metrics_test])
display(metrics_df)
print(
    "If the model predicts only one class: specificity or sensitivity hits 0, "
    "f1_benign or f1_malignant hits 0, and f1_macro stays low even if accuracy looks ok."
)
if SMOKE_TEST:
    print("SMOKE_TEST=True: this is a tiny wiring test. Treat the numbers as a smoke-test check only.")


def inspect_case(split_name, df, i):
    row = df.iloc[i]
    img = Term("tensor", Term(split_name, Constant(i)))
    queries = {
        "malignant": Query(Term("malignant", img)),
        "kind_mass": Query(Term("lesion_kind", img, Term("mass"))),
        "kind_calc": Query(Term("lesion_kind", img, Term("calcification"))),
        "shape_irregular": Query(Term("shape", img, Term("irregular"))),
        "margin_spiculated": Query(Term("margin", img, Term("spiculated"))),
        "calc_suspicious": Query(Term("calc", img, Term("suspicious"))),
    }
    print(f"\n[{split_name} #{i}] patient={row['patient_id']} pathology={row['pathology']} kind={row['abnormality_kind']}")
    print(f"  gold: shape={row['shape_concept']} margin={row['margin_concept']} calc={row['calc_concept']} kind={row.get('kind_concept')}")
    if "resolved_path" in row.index:
        print(f"  image: {row['resolved_path']}")
    for name, q in queries.items():
        solved = model.solve([q])
        print(f"  P({name}) = {extract_prob(solved[0]) if solved else 0.0:.3f}")


print("Example reasoning traces (validation):")
for i in range(min(3, len(val_set))):
    inspect_case("val", df_val, i)


def gold_concept_prob(split_name, i, predicate, label):
    q = Query(Term(predicate, Term("tensor", Term(split_name, Constant(i))), Term(label)))
    solved = model.solve([q])
    return extract_prob(solved[0]) if solved else 0.0


print("\nValidation concept check (mean P(gold label); >0.5 means the net usually agrees with the CSV):")
for pred, col in (
    ("lesion_kind", "kind_concept"),
    ("shape", "shape_concept"),
    ("margin", "margin_concept"),
    ("calc", "calc_concept"),
):
    probs = [gold_concept_prob("val", i, pred, df_val.iloc[i][col]) for i in range(len(df_val))]
    arr = np.array(probs)
    print(f"  {pred:12s}  mean P(gold)={arr.mean():.3f}  agree@0.5={(arr >= 0.5).mean():.3f}")


def show_prediction_grid(title, df, tensors, y_prob, n=None):
    n = min(int(n if n is not None else SHOW_N_IMAGES), len(df), len(tensors))
    y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        ax = axes[i]
        ax.imshow(tensor_to_numpy(tensors[i]), cmap="gray")
        row = df.iloc[i]
        gold = "MAL" if int(row["malignant"]) == 1 else "BEN"
        pred = "MAL" if int(y_pred[i]) == 1 else "BEN"
        ok = gold == pred
        ax.set_title(
            f"{row['patient_id']}  gold={gold} pred={pred}\nP={y_prob[i]:.2f}  {row['abnormality_kind']}",
            fontsize=8,
            color=("green" if ok else "red"),
        )
        ax.axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


print("\nPredicted test / validation images (green = match, red = mismatch):")
show_prediction_grid("Validation images + predictions", df_val, val_tensors, val_prob)
show_prediction_grid("Test images + predictions", df_test, test_tensors, test_prob)

## 12) Save weights and metrics

**What:** store the model snapshot and a metrics JSON under `OUTPUT_DIR`.

**Why:** so you can reload this run later without training again.


In [ ]:
snapshot = OUTPUT_DIR / ("deepproblog_smoke.pth" if SMOKE_TEST else "deepproblog_model.pth")
model.save_state(str(snapshot))
payload = {
    "smoke_test": SMOKE_TEST,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "n_train": len(train_set),
    "n_val": len(val_set),
    "n_test": len(test_set),
    "metrics_val": metrics_val,
    "metrics_test": metrics_test,
    "note": "Smoke-test numbers are a wiring check only. Rules are a computational program, not official BI-RADS.",
}
(OUTPUT_DIR / "deepproblog_metrics.json").write_text(json.dumps(payload, indent=2))
print("Saved:", snapshot)
print("Saved:", OUTPUT_DIR / "deepproblog_metrics.json")
print("Done. Single-notebook pipeline complete.")